In [10]:
sample = r'''low low low low low
lower lower widest widest widest
newest newest newest newest newest newest'''
with open("data/sample.txt", "w", encoding="utf-8") as f:
    f.write(sample)

In [2]:
def load_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()
text = load_text("data/TinyStoriesV2-GPT4-valid.txt")

In [3]:
import regex as re
import multiprocessing

multiprocessing.set_start_method("fork", force=True)

special_tokens = ["<|endoftext|>"]
pattern = "|".join(re.escape(tok) for tok in special_tokens)
chunks = re.split(pattern, text)

def pretokenize(chunk):
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    return re.findall(PAT, chunk)

with multiprocessing.Pool() as pool:
    results = pool.map(pretokenize, chunks)

for chunk, tokens in zip(chunks, results):
    print(chunk)
    print(tokens)
    break

u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.

['u', ' don', "'t", ' have', ' to', ' be', ' scared', ' of', ' the', ' loud', ' dog', ',', ' I', "'ll", ' protect', ' you', '".', ' The', ' mole', ' felt', ' so', ' safe', ' with', ' the', ' little', ' girl', '.', ' She', ' was', ' very', ' kind', ' and', ' the', ' mole', ' soon', ' came', ' to', ' trust', ' her', '.', ' He', ' leaned', ' against', ' her', ' and', ' she', ' kept', ' him', ' safe', '.', ' The', ' mole', ' had', ' found', ' his', ' best', ' friend', '.', '\n']


In [ ]:
import regex as re
from collections import Counter, defaultdict
sample = r'''low low low low low
lower lower widest widest widest
newest newest newest newest newest newest'''
with open("data/sample.txt", "w", encoding="utf-8") as f:
    f.write(sample)

results = [pretokenize(sample)]

merge = []
vocab = [bytes([i]) for i in range(256)]
vocab.append(special_tokens)

token_counter = defaultdict(int)
for tokens in results:
    for token in tokens:
        utf8_encoded = token.encode("utf-8")
        key = tuple(bytes([x]) for x in utf8_encoded) # key is a tuple of bytes 
        token_counter[key] += 1
        

In [5]:
pair_counter = defaultdict(int)
for token, count in token_counter.items():
    for i in range(len(token)-1):
        pair = token[i:i+2]
        pair_counter[pair] += count

In [6]:
for _ in range(1):
    max_pair = max(pair_counter.keys(), key = lambda pair: (pair_counter[pair], pair))
    max_pair_count = pair_counter[max_pair]
    merged_pair = max_pair[0] + max_pair[1]
    merge.append(max_pair)
    new_counter = {}
    for token, count in token_counter.items():
        new_token = []
        i = 0
        while i < len(token):
            pair = token[i:i+2]
            if pair == max_pair:
                # Update the pair_counts
                pair_counter[pair] -= count
                if i > 0:
                    left_pair = (new_token[-1],pair[0])
                    pair_counter[left_pair] -= count
                    pair_counter[(left_pair[0], merged_pair)] += count
                if i < len(token)-2:
                    right_pair = token[i+1:i+3]
                    pair_counter[right_pair] -= count
                    pair_counter[(merged_pair, right_pair[1])] += count
                new_token.append(merged_pair)
                i += 2
            else:
                new_token.append(token[i])
                i += 1
        new_counter[tuple(new_token)] = count
    token_counter = new_counter
    vocab.append(merged_pair)

In [7]:
token_counter

{(b'l', b'o', b'w'): 1,
 (b' ', b'l', b'o', b'w'): 4,
 (b'\n',): 2,
 (b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'l', b'o', b'w', b'e', b'r'): 1,
 (b' ', b'w', b'i', b'd', b'e', b'st'): 3,
 (b'n', b'e', b'w', b'e', b'st'): 1,
 (b' ', b'n', b'e', b'w', b'e', b'st'): 5}

In [8]:
pair_counter

defaultdict(int,
            {(b'l', b'o'): 7,
             (b'o', b'w'): 7,
             (b' ', b'l'): 5,
             (b'w', b'e'): 8,
             (b'e', b'r'): 2,
             (b' ', b'w'): 3,
             (b'w', b'i'): 3,
             (b'i', b'd'): 3,
             (b'd', b'e'): 3,
             (b'e', b's'): 0,
             (b's', b't'): 0,
             (b'n', b'e'): 6,
             (b'e', b'w'): 6,
             (b' ', b'n'): 5,
             (b'e', b'st'): 9})

In [ ]:
import regex as re
import multiprocessing
multiprocessing.set_start_method("fork", force=True)

special_tokens = ["<|endoftext|>"]
input_path = "data/TinyStoriesV2-GPT4-valid.txt"
vocab_size = 1000000

def train_bpe(input_path, vocab_size, special_tokens):
    merge = []
    vocab = [bytes([i]) for i in range(256)]
    vocab.append(special_tokens)

    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()

    pattern = "|".join(re.escape(tok) for tok in special_tokens)
    chunks = re.split(pattern, text)

    def pretokenize(chunk):
        PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        return re.findall(PAT, chunk)
    with multiprocessing.Pool() as pool:
        pretokenized_chunks = pool.map(pretokenize, chunks)

    token_counter = defaultdict(int)
    for tokens in pretokenized_chunks:
        for token in tokens:
            utf8_encoded = token.encode("utf-8")
            key = tuple(bytes([x]) for x in utf8_encoded) # key is a tuple of bytes 
            token_counter[key] += 1


    pair_counter = defaultdict(int)
    for token, count in token_counter.items():
        for i in range(len(token)-1):
            pair = token[i:i+2]
            pair_counter[pair] += count


    while len(vocab) < vocab_size:
        max_pair = max(pair_counter.keys(), key = lambda pair: (pair_counter[pair], pair))
        max_pair_count = pair_counter[max_pair]
        merged_pair = max_pair[0] + max_pair[1]
        merge.append(max_pair)
        new_counter = {}
        for token, count in token_counter.items():
            new_token = []
            i = 0
            while i < len(token):
                pair = token[i:i+2]
                if pair == max_pair:
                    # Update the pair_counts
                    pair_counter[pair] -= count
                    if i > 0:
                        left_pair = (new_token[-1],pair[0])
                        pair_counter[left_pair] -= count
                        pair_counter[(left_pair[0], merged_pair)] += count
                    if i < len(token)-2:
                        right_pair = token[i+1:i+3]
                        pair_counter[right_pair] -= count
                        pair_counter[(merged_pair, right_pair[1])] += count
                    new_token.append(merged_pair)
                    i += 2
                else:
                    new_token.append(token[i])
                    i += 1
            new_counter[tuple(new_token)] = count
        token_counter = new_counter
        vocab.append(merged_pair)

    return merge, vocab
        
        
    
    
        
    
    